In [ ]:

import hashlib
import itertools
import string
from typing import Dict, Set
import time
import os

def sha256_hash(password: str) -> str:
    """
    Compute SHA-256 hash of a UTF-8 encoded password.
    
    Parameters:
        password (str): Plain text password
        
    Returns:
        str: Hexadecimal SHA-256 hash (64 characters)
    """
    return hashlib.sha256(password.encode('utf-8')).hexdigest()


# Target hashes from the problem
TARGET_HASHES = {
    '5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8': 'Hash 1',
    '873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34': 'Hash 2',
    'b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342': 'Hash 3'
}

print("Target Hashes:")
for hash_val, hash_id in TARGET_HASHES.items():
    print(f"  {hash_id}: {hash_val}")
print()


def load_wordlist_from_file(filename: str, max_lines: int = 100000) -> list:
    """
    Load a password wordlist from a local file.
    
    Parameters:
        filename (str): Local filename to read from
        max_lines (int): Maximum lines to read
        
    Returns:
        list: Password list
    """
    passwords = []
    
    if os.path.exists(filename):
        print(f"  Loading wordlist from file: {filename}")
        try:
            with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
                for i, line in enumerate(f):
                    if i >= max_lines:
                        break
                    pwd = line.strip()
                    if pwd:
                        passwords.append(pwd)
            print(f"  Loaded {len(passwords):,} passwords from file")
            return passwords
        except Exception as e:
            print(f"  Error reading file: {e}")
    
    print(f"  File {filename} not found or could not be read.")
    return []


def load_common_passwords() -> list:
    """
    Load comprehensive password wordlists from multiple sources.
    
    Returns:
        list: Comprehensive password list
    """
    all_passwords = []
    
    print("Strategy 1: Dictionary Attack with Real-World Wordlists")
    print("-" * 70)
    print()
    
    # Load local wordlist files
    # Note: Replace 'local_wordlist_file.txt' with the correct path to your uploaded file
    wordlist_file = 'rockyou-75.txt'
    local_passwords = load_wordlist_from_file(wordlist_file, max_lines=100000)
    all_passwords.extend(local_passwords)
    print()

    # Also generate short passwords (very common in real world)
    print("Source 2: Generating all 1-4 character passwords")
    print("  These represent ~50% of weak passwords in practice")
    short_passwords = []
    for length in range(1, 5):
        for combo in itertools.product(string.ascii_lowercase, repeat=length):
            short_passwords.append(''.join(combo))
    
    all_passwords.extend(short_passwords)
    print(f"  Generated {len(short_passwords):,} short passwords")
    print()
    
    # Remove duplicates
    unique_passwords = list(dict.fromkeys(all_passwords))
    
    print(f"Total unique passwords in dictionary: {len(unique_passwords):,}")
    print()
    
    return unique_passwords


def dictionary_attack(target_hashes: Dict[str, str]) -> Dict[str, str]:
    """
    Dictionary attack using real-world password datasets.
    
    Uses passwords from:
    - SecLists (GitHub - Daniel Miessler)
    - RockYou breach database
    - Short password combinations
    
    Parameters:
        target_hashes (dict): Hash to identifier mapping
        
    Returns:
        dict: Cracked passwords
    """
    cracked = {}
    
    print("=" * 70)
    print("EXECUTING DICTIONARY ATTACK")
    print("=" * 70)
    print()
    
    wordlist = load_common_passwords()
    
    print("Testing passwords against target hashes...")
    print()
    
    start_time = time.time()
    tested = 0
    
    for password in wordlist:
        computed_hash = sha256_hash(password)
        
        if computed_hash in target_hashes:
            hash_id = target_hashes[computed_hash]
            cracked[hash_id] = password
            print(f"  FOUND {hash_id}: '{password}'")
        
        tested += 1
        
        # Progress indicator
        if tested % 50000 == 0:
            print(f"  Tested {tested:,} passwords...", end='\r', flush=True)
    
    elapsed = time.time() - start_time

    print()
    print(f"\nDictionary attack complete:")
    print(f"  Tested: {tested:,} passwords")
    print(f"  Time: {elapsed:.2f} seconds")
    print(f"  Speed: {tested/elapsed:,.0f} passwords/second")
    print(f"  Cracked: {len(cracked)}/{len(target_hashes)} hashes")
    
    return cracked


def systematic_brute_force(target_hashes: Dict[str, str], 
                          already_found: Set[str],
                          max_overall_length: int = 10) -> Dict[str, str]:
    """
    Systematic brute force with progressively expanding search.
    
    This approach guarantees finding any password within the search space
    by testing all possible combinations in a logical order.
    
    Parameters:
        target_hashes (dict): Hash to identifier mapping
        already_found (set): Hash IDs already cracked
        max_overall_length (int): Maximum password length to attempt
        
    Returns:
        dict: Newly cracked passwords
    """
    cracked = {}
    remaining = {h: i for h, i in target_hashes.items() if i not in already_found}
    
    if not remaining:
        return cracked
    
    print()
    print("=" * 70)
    print("EXECUTING BRUTE FORCE ATTACK")
    print("=" * 70)
    print()
    print("Systematically testing all combinations...")
    print("Character sets ordered by real-world password frequency")
    print()
    
    # Character sets ordered by frequency in real passwords
    # Based on analysis of breach databases
    strategies = [
        ("lowercase", string.ascii_lowercase, 10),
        ("digits", string.digits, 12),
        ("uppercase", string.ascii_uppercase, 8),
        ("lower+digit", string.ascii_lowercase + string.digits, 7),
        ("letters", string.ascii_letters, 6),
        ("alphanum", string.ascii_letters + string.digits, 5),
    ]
    
    for strategy_name, charset, max_len in strategies:
        if not remaining:
            print("\n All passwords found!")
            break
        
        # Don't exceed overall max length
        max_len = min(max_len, max_overall_length)
        
        print(f"\nCharacter set: {strategy_name}")
        print(f"  Alphabet: {charset}")
        print(f"  Size: {len(charset)} characters")
        print(f"  Searching up to length: {max_len}")
        print()
        
        for length in range(1, max_len + 1):
            if not remaining:
                break
            
            total_combinations = len(charset) ** length
            print(f"  Length {length}: {total_combinations:,} combinations...", end=' ', flush=True)
            
            start_time = time.time()
            tested = 0
            
            for password_tuple in itertools.product(charset, repeat=length):
                password = ''.join(password_tuple)
                computed_hash = sha256_hash(password)
                
                if computed_hash in remaining:
                    hash_id = remaining[computed_hash]
                    cracked[hash_id] = password
                    elapsed = time.time() - start_time
                    print(f"\n    FOUND {hash_id}: '{password}' (after {tested:,} attempts in {elapsed:.2f}s)")
                    del remaining[hash_id]
                    
                    if not remaining:
                        return cracked
                    
                    print(f"  Length {length}: continuing...", end=' ', flush=True)
                
                tested += 1
                
                # Progress dots every 1M attempts
                if tested % 1000000 == 0:
                    print('.', end='', flush=True)
            
            elapsed = time.time() - start_time
            print(f"complete ({elapsed:.2f}s)")
    
    return cracked


# Execute the comprehensive attack
start_total = time.time()

# Phase 1: Dictionary Attack with real wordlists
results = dictionary_attack(TARGET_HASHES)

# Phase 2: Systematic Brute Force (if needed)
if len(results) < len(TARGET_HASHES):
    brute_results = systematic_brute_force(
        TARGET_HASHES, 
        set(results.keys()),
        max_overall_length=10
    )
    results.update(brute_results)

elapsed_total = time.time() - start_total

# Display Final Results
print()
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print()

for hash_val, hash_id in TARGET_HASHES.items():
    password = results.get(hash_id, "NOT FOUND")
    
    print(f"{hash_id}:")
    print(f"  Hash:     {hash_val}")
    print(f"  Password: '{password}'")
    
    if password != "NOT FOUND":
        # Verify
        computed = sha256_hash(password)
        is_correct = computed == hash_val
        print(f"  Length:   {len(password)} characters")
        
        # Character analysis
        char_info = []
        if any(c.islower() for c in password):
            char_info.append("lowercase")
        if any(c.isupper() for c in password):
            char_info.append("uppercase")
        if any(c.isdigit() for c in password):
            char_info.append("digits")
        
        print(f"  Contains: {', '.join(char_info)}")
    else:
        print(f"  Status:   NOT CRACKED")
    
    print()

print(f"Success Rate: {len(results)}/{len(TARGET_HASHES)} passwords")
print(f"Total Time: {elapsed_total:.2f} seconds")
print()


In [ ]:
"""How it works:

verify_password() Function: This function takes a password and a given hash, computes the SHA-256 hash of the password, and compares it with the given hash. 
                            It returns True if the hashes match and False otherwise.

passwords_to_check List: This list holds tuples where each tuple contains a cracked password and its corresponding hash. 
"""

import hashlib

def verify_password(cracked_password, given_hash):
    """ Function to compute the hash of the cracked password and compare with the given hash """
    computed_hash = hashlib.sha256(cracked_password.encode('utf-8')).hexdigest()
    return computed_hash == given_hash

# Define cracked passwords and their corresponding hashes
passwords_to_check = [
    ("password", '5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8'),  # Hash of 'password'
    ("cheese", '873ac9ffea4dd04fa719e8920cd6938f0c23cd678af330939cff53c3d2855f34'),   # Hash of 'cheese'
    ("P@ssw0rd", 'b03ddf3ca2e714a6548e7495e2a03f5e824eaac9837cd7f159c67b90fb4b7342') # Hash of 'P@ssw0rd'
]

# Check each password and verify
for password, given_hash in passwords_to_check:
    if verify_password(password, given_hash):
        print(f"The cracked password '{password}' matches the given hash!")
    else:
        print(f"The cracked password '{password}' does NOT match the given hash.")

In [ ]:
# ****************************************************************************



































# ****************************************************************************